# Dog Cat Image Classifier

**Topic:** Dog vs Cat Image Classification  
**Framework:** TensorFlow / Keras  

## Imports and Reproducibility

The following libraries are used throughout the notebook.


In [ ]:
import os
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))


# Dataset Preparation

The dataset is downloaded from the provided GitHub repository.  
The training folder is split into **80% training** and **20% validation** using `validation_split=0.2`.

All images are resized to **224 × 224** and normalized to the range **[0, 1]**.


In [ ]:
# Download the dataset from the repository provided in the question.
# The command runs only if the folder does not already exist.

if not os.path.exists("/content/dog-cat-full-dataset"):
    !git clone https://github.com/laxmimerit/dog-cat-full-dataset.git
else:
    print("Dataset already downloaded.")


In [ ]:
DATA_ROOT = "/content/dog-cat-full-dataset/data"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="binary",
)

class_names = train_ds.class_names
print("Class names:", class_names)


### Normalize images to [0, 1]

The raw image pixels are in the range 0–255.  
`Rescaling(1./255)` converts them to 0–1.


In [ ]:
normalization_layer = layers.Rescaling(1.0 / 255)

train_ds = train_ds.map(
    lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE
)
val_ds = val_ds.map(
    lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE
)
test_ds = test_ds.map(
    lambda x, y: (normalization_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE
)


### Dataset optimization with `prefetch()`

`prefetch(tf.data.AUTOTUNE)` prepares future batches while the model is training on the current batch, improving input pipeline efficiency.


In [ ]:
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)


In [ ]:
def count_samples(dataset):
    total = 0
    for _, labels in dataset:
        total += labels.shape[0]
    return int(total)


train_count = count_samples(train_ds)
val_count = count_samples(val_ds)
test_count = count_samples(test_ds)

print(f"Training samples   : {train_count}")
print(f"Validation samples : {val_count}")
print(f"Test samples       : {test_count}")
print(f"Classes            : {class_names}")


In [ ]:
plt.figure(figsize=(10, 8))
for images, labels in train_ds.take(1):
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy())
        label_index = int(labels[i].numpy().squeeze())
        plt.title(class_names[label_index])
        plt.axis("off")
plt.tight_layout()
plt.show()


# Data Augmentation 

The augmentation pipeline contains:

- Random Flip
- Random Rotation
- Random Zoom
- Random Contrast as an additional augmentation

The pipeline is placed inside the training models. Keras augmentation layers operate only during training and are automatically disabled during validation/testing/inference.


In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.10),
        layers.RandomContrast(0.10),
    ],
    name="data_augmentation",
)

data_augmentation


### Original image vs augmented images


In [ ]:
for images, labels in train_ds.take(1):
    sample_image = images[0]
    sample_label = int(labels[0].numpy().squeeze())

    plt.figure(figsize=(12, 7))

    ax = plt.subplot(2, 3, 1)
    plt.imshow(sample_image.numpy())
    plt.title(f"Original: {class_names[sample_label]}")
    plt.axis("off")

    for i in range(5):
        augmented = data_augmentation(tf.expand_dims(sample_image, 0), training=True)[0]

        ax = plt.subplot(2, 3, i + 2)
        plt.imshow(tf.clip_by_value(augmented, 0, 1).numpy())
        plt.title(f"Augmented {i + 1}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Common Utilities

The next helper functions are reused for training, plotting, evaluation, and comparison.


In [ ]:
def make_early_stopping():
    return tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )


def plot_history(history, title_prefix="Model"):
    epochs = range(1, len(history.history["loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history["accuracy"], label="Training Accuracy")
    plt.plot(epochs, history.history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix}: Training vs Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history.history["loss"], label="Training Loss")
    plt.plot(epochs, history.history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix}: Training vs Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


def final_history_values(history):
    return {
        "training_loss": history.history["loss"][-1],
        "training_accuracy": history.history["accuracy"][-1],
        "validation_loss": history.history["val_loss"][-1],
        "validation_accuracy": history.history["val_accuracy"][-1],
    }


def evaluate_binary_model(model, dataset):
    test_loss, test_accuracy = model.evaluate(dataset, verbose=0)

    y_true = []
    y_prob = []

    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0).reshape(-1)
        y_prob.extend(probabilities.tolist())
        y_true.extend(labels.numpy().reshape(-1).astype(int).tolist())

    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return {
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "test_precision": float(precision),
        "test_recall": float(recall),
        "test_f1": float(f1),
    }


def total_params(model):
    return model.count_params()


# Custom CNN Model

The custom CNN contains three convolutional blocks. Each block uses:

1. Convolution
2. Batch Normalization
3. ReLU activation
4. Max Pooling

Dropout is used for regularization, followed by a dense classification head and a sigmoid output for binary classification.


In [ ]:
def build_custom_cnn():
    inputs = keras.Input(shape=(224, 224, 3))

    x = data_augmentation(inputs)

    # Convolutional Block 1
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Convolutional Block 2
    x = layers.Conv2D(64, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    # Convolutional Block 3
    x = layers.Conv2D(128, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Dropout(0.30)(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Dense classification head
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(1, activation="sigmoid")(x)

    return keras.Model(inputs, outputs, name="Custom_CNN")


custom_model = build_custom_cnn()

custom_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

custom_model.summary()


In [ ]:
custom_early_stopping = make_early_stopping()

custom_history = custom_model.fit(
    train_ds, validation_data=val_ds, epochs=20, callbacks=[custom_early_stopping]
)


In [ ]:
plot_history(custom_history, "Custom CNN")


In [ ]:
custom_metrics = evaluate_binary_model(custom_model, test_ds)

print("Custom CNN Test Results")
for key, value in custom_metrics.items():
    print(f"{key}: {value:.4f}")


# Transfer Learning / Feature Extraction 

For feature extraction, **MobileNetV2** pretrained on ImageNet is used.

The original classifier is removed by setting `include_top=False`.  
The backbone is frozen using `base_model.trainable = False`.  
Only the newly added classifier is trained.

Because this notebook normalized the dataset to **[0, 1]**, the model converts images back to the range expected by MobileNetV2 before applying `preprocess_input`.


In [ ]:
def build_feature_extraction_model():
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)

    # Dataset is [0,1]. Convert to [0,255] before MobileNetV2 preprocessing.
    x = layers.Lambda(
        lambda z: tf.keras.applications.mobilenet_v2.preprocess_input(z * 255.0),
        name="mobilenet_preprocessing",
    )(x)

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs, name="MobileNetV2_Feature_Extraction")
    return model, base_model


feature_model, feature_backbone = build_feature_extraction_model()

feature_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

feature_model.summary()

print("Backbone trainable:", feature_backbone.trainable)
print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in feature_backbone.layers),
)


In [ ]:
feature_early_stopping = make_early_stopping()

feature_history = feature_model.fit(
    train_ds, validation_data=val_ds, epochs=15, callbacks=[feature_early_stopping]
)


In [ ]:
plot_history(feature_history, "MobileNetV2 Feature Extraction")


In [ ]:
feature_metrics = evaluate_binary_model(feature_model, test_ds)

print("Feature Extraction Test Results")
for key, value in feature_metrics.items():
    print(f"{key}: {value:.4f}")


### Why is freezing the backbone useful?

Freezing the pretrained backbone is useful because:

1. The ImageNet-trained convolutional layers already contain useful low-level and high-level visual features such as edges, textures, shapes, and object parts.
2. Freezing prevents these useful pretrained weights from being destroyed during the early stages of training.
3. Only the small new classifier must be optimized, so training becomes faster and requires less GPU memory.
4. It also reduces the risk of overfitting when the new dataset is smaller than ImageNet.

This approach is called **feature extraction** or **linear probing** because the pretrained feature extractor remains fixed while a new classifier is learned.


# Full Fine-Tuning

A different pretrained architecture is required, so this section uses **ResNet50**.

Unlike feature extraction, all backbone layers are unfrozen and the entire network is trained.  
A much smaller learning rate is used to avoid making destructive changes to pretrained weights.


In [ ]:
def build_finetuning_model():
    base_model = tf.keras.applications.ResNet50(
        input_shape=(224, 224, 3), include_top=False, weights="imagenet"
    )

    # Full fine-tuning: unfreeze every layer
    base_model.trainable = True

    inputs = keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)

    # Dataset is [0,1]. Convert to [0,255] before ResNet50 preprocessing.
    x = layers.Lambda(
        lambda z: tf.keras.applications.resnet50.preprocess_input(z * 255.0),
        name="resnet_preprocessing",
    )(x)

    x = base_model(x, training=True)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs, outputs, name="ResNet50_Full_Fine_Tuning")
    return model, base_model


finetune_model, finetune_backbone = build_finetuning_model()

finetune_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

finetune_model.summary()

print("Backbone trainable:", finetune_backbone.trainable)
print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in finetune_backbone.layers),
)


In [ ]:
finetune_early_stopping = make_early_stopping()

finetune_history = finetune_model.fit(
    train_ds, validation_data=val_ds, epochs=15, callbacks=[finetune_early_stopping]
)


In [ ]:
plot_history(finetune_history, "ResNet50 Full Fine-Tuning")


In [ ]:
finetune_metrics = evaluate_binary_model(finetune_model, test_ds)

print("Fine-Tuned ResNet50 Test Results")
for key, value in finetune_metrics.items():
    print(f"{key}: {value:.4f}")


### Difference between feature extraction and full fine-tuning

**Feature Extraction**
- The pretrained backbone is frozen.
- Only the newly added classifier learns.
- Training is faster and computationally cheaper.
- It is often a strong choice when the new dataset is relatively small.
- There is a lower risk of destroying useful pretrained features.

**Full Fine-Tuning**
- The pretrained backbone is unfrozen.
- Both the backbone and classifier learn from the new dataset.
- Training is slower and uses more memory/compute.
- It can adapt the feature representation more strongly to the new task.
- A small learning rate is usually required to protect pretrained knowledge.


# Model Comparison

The following section compares:

1. Custom CNN
2. MobileNetV2 Feature Extraction
3. ResNet50 Full Fine-Tuning

The table includes all metrics requested in the assignment.


In [ ]:
def build_comparison_row(name, model, history, test_metrics):
    hist = final_history_values(history)

    return {
        "Model": name,
        "Total Parameters": total_params(model),
        "Training Loss": hist["training_loss"],
        "Training Accuracy": hist["training_accuracy"],
        "Validation Loss": hist["validation_loss"],
        "Validation Accuracy": hist["validation_accuracy"],
        "Test Loss": test_metrics["test_loss"],
        "Test Accuracy": test_metrics["test_accuracy"],
        "Test Precision": test_metrics["test_precision"],
        "Test Recall": test_metrics["test_recall"],
        "Test F1 Score": test_metrics["test_f1"],
    }


comparison_df = pd.DataFrame(
    [
        build_comparison_row(
            "Custom CNN", custom_model, custom_history, custom_metrics
        ),
        build_comparison_row(
            "MobileNetV2 Feature Extraction",
            feature_model,
            feature_history,
            feature_metrics,
        ),
        build_comparison_row(
            "ResNet50 Full Fine-Tuning",
            finetune_model,
            finetune_history,
            finetune_metrics,
        ),
    ]
)

comparison_df


In [ ]:
display_df = comparison_df.copy()

metric_columns = [
    "Training Loss",
    "Training Accuracy",
    "Validation Loss",
    "Validation Accuracy",
    "Test Loss",
    "Test Accuracy",
    "Test Precision",
    "Test Recall",
    "Test F1 Score",
]

display_df[metric_columns] = display_df[metric_columns].round(4)
display_df


In [ ]:
best_accuracy_model = comparison_df.loc[
    comparison_df["Test Accuracy"].idxmax(), "Model"
]

best_f1_model = comparison_df.loc[comparison_df["Test F1 Score"].idxmax(), "Model"]

fewest_params_model = comparison_df.loc[
    comparison_df["Total Parameters"].idxmin(), "Model"
]

print("Best Test Accuracy :", best_accuracy_model)
print("Best Test F1 Score :", best_f1_model)
print("Fewest Parameters  :", fewest_params_model)


### Discussion

**Which model performed best?**  
The best-performing model should be identified from the generated comparison table using **test accuracy** and **test F1 score**. The code above automatically prints the model with the highest values.

**Which model is most computationally efficient?**  
The custom CNN normally has far fewer parameters than large pretrained architectures, but MobileNetV2 is specifically designed to be lightweight and efficient. The final judgment should consider not only total parameters but also training time, inference speed, and whether the backbone is frozen.

**Which model would be deployed in production and why?**  
A strong production choice is often **MobileNetV2 feature extraction** when accuracy is competitive because it provides a good balance between predictive performance, model size, inference speed, and computational cost. However, if the fine-tuned ResNet50 achieves a substantially better test score and the deployment environment has sufficient compute resources, the fine-tuned model may be preferred.


# Error Analysis

The reusable `error_analysis()` function:

- Predicts the test dataset
- Prints a classification report
- Generates a confusion matrix
- Displays at least 9 incorrectly classified images
- Shows actual label, predicted label, and prediction confidence


In [ ]:
def error_analysis(model, test_dataset, class_names, num_errors=9):
    y_true = []
    y_pred = []
    y_prob = []
    all_images = []

    for images, labels in test_dataset:
        probabilities = model.predict(images, verbose=0).reshape(-1)
        predictions = (probabilities >= 0.5).astype(int)

        all_images.extend(images.numpy())
        y_true.extend(labels.numpy().reshape(-1).astype(int))
        y_pred.extend(predictions.tolist())
        y_prob.extend(probabilities.tolist())

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)
    all_images = np.array(all_images)

    print("CLASSIFICATION REPORT")
    print(
        classification_report(
            y_true, y_pred, target_names=class_names, digits=4, zero_division=0
        )
    )

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm, cmap="Blues")
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.xticks(range(len(class_names)), class_names)
    plt.yticks(range(len(class_names)), class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.colorbar()
    plt.tight_layout()
    plt.show()

    incorrect_indices = np.where(y_true != y_pred)[0]

    print(f"Total incorrect predictions: {len(incorrect_indices)}")

    if len(incorrect_indices) == 0:
        print("No incorrectly classified images were found.")
        return

    n = min(num_errors, len(incorrect_indices))
    selected_indices = incorrect_indices[:n]

    rows = int(np.ceil(n / 3))
    plt.figure(figsize=(12, 4 * rows))

    for plot_index, data_index in enumerate(selected_indices):
        actual = int(y_true[data_index])
        predicted = int(y_pred[data_index])
        probability_dog = float(y_prob[data_index])

        # Confidence of the predicted class
        confidence = probability_dog if predicted == 1 else 1.0 - probability_dog

        ax = plt.subplot(rows, 3, plot_index + 1)
        plt.imshow(np.clip(all_images[data_index], 0, 1))
        plt.title(
            f"Actual: {class_names[actual]}\n"
            f"Predicted: {class_names[predicted]}\n"
            f"Confidence: {confidence:.2%}"
        )
        plt.axis("off")

    plt.tight_layout()
    plt.show()


### Run error analysis on the best model

You may change `best_model_for_analysis` to any of the three trained models.


In [ ]:
best_model_for_analysis = finetune_model

error_analysis(best_model_for_analysis, test_ds, class_names, num_errors=9)


# Final Conclusion

This project implemented three approaches to binary dog-vs-cat image classification:

- A CNN built from scratch
- Transfer learning using a frozen MobileNetV2 backbone
- Full fine-tuning using ResNet50

The models were trained with data augmentation and EarlyStopping. Their performance was compared using loss, accuracy, precision, recall, and F1 score. Finally, error analysis was performed using a classification report, confusion matrix, and incorrectly classified test images.

> Run all training cells before using the comparison and error-analysis sections so that the notebook contains the actual experimental results.
